# Submit Parent Order

Professional parent-order submission report for the Adaptive Quantum SOR API. Use the widget panel to configure the order, then submit it and review a clear result report.

## 1. Order Inputs

Use the controls below to configure the parent order payload and API endpoint.

In [ ]:
from datetime import datetime, timezone
from html import escape
from IPython.display import HTML, display
import requests
import pandas as pd

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    widgets = None

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)

REPORT_CSS = """
<style>
.sor-report {font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; color:#111827;}
.sor-hero {border:1px solid #d7dde6;border-radius:8px;padding:18px 20px;margin:4px 0 18px 0;background:linear-gradient(180deg,#ffffff 0%,#f8fafc 100%);}
.sor-hero h1 {font-size:24px;margin:0 0 6px 0;letter-spacing:0;}
.sor-hero p {margin:0;color:#5b6472;font-size:14px;}
.sor-section {margin:22px 0 10px 0;}
.sor-section h2 {font-size:18px;margin:0 0 4px 0;letter-spacing:0;}
.sor-section p {margin:0 0 8px 0;color:#5b6472;font-size:13px;}
.sor-kpi-grid {display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;margin:10px 0 18px 0;}
.sor-kpi {border:1px solid #d7dde6;border-radius:8px;padding:12px 14px;background:#ffffff;}
.sor-kpi span {display:block;color:#5b6472;font-size:12px;margin-bottom:5px;}
.sor-kpi strong {display:block;color:#111827;font-size:22px;line-height:1.15;}
.sor-note {border-left:4px solid #1d4ed8;background:#eef5ff;padding:10px 12px;margin:10px 0 18px 0;color:#1e3a8a;}
.sor-badge {display:inline-block;border-radius:999px;padding:3px 9px;font-size:12px;font-weight:700;}
.sor-pass {background:#ecfdf5;color:#047857;border:1px solid #a7f3d0;}
.sor-warn {background:#fffbeb;color:#b45309;border:1px solid #fde68a;}
.sor-fail {background:#fef2f2;color:#b91c1c;border:1px solid #fecaca;}
table.sor-report-table {border-collapse:collapse;width:100%;table-layout:auto;margin:8px 0 18px 0;font-size:13px;}
table.sor-report-table th, table.sor-report-table td {border:1px solid #d7dde6;padding:7px 9px;vertical-align:top;white-space:normal;overflow-wrap:anywhere;word-break:break-word;max-width:560px;text-align:left;}
table.sor-report-table th {background:#f8fafc;color:#111827;font-weight:700;}
table.sor-report-table tbody tr:nth-child(even) {background:#fcfdff;}
.sor-bar-panel {border:1px solid #d7dde6;border-radius:8px;padding:12px;background:#ffffff;margin:8px 0 18px 0;}
.sor-bar-row {display:grid;grid-template-columns:220px 1fr 100px;gap:10px;align-items:center;margin:7px 0;}
.sor-bar-label {font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;color:#374151;}
.sor-bar-track {height:12px;background:#eef2f7;border-radius:3px;overflow:hidden;}
.sor-bar {height:12px;background:#2563eb;}
.sor-bar-value {text-align:right;font-variant-numeric:tabular-nums;color:#111827;}
</style>
"""

def display_styles():
    display(HTML(REPORT_CSS))

def display_header(title, subtitle, generated_at):
    display(HTML(f"""
    <div class="sor-report sor-hero">
      <h1>{escape(title)}</h1>
      <p>{escape(subtitle)}</p>
      <p>Generated at {escape(generated_at)} UTC</p>
    </div>
    """))

def display_section(title, subtitle=""):
    subtitle_html = f"<p>{escape(subtitle)}</p>" if subtitle else ""
    display(HTML(f"<div class='sor-report sor-section'><h2>{escape(title)}</h2>{subtitle_html}</div>"))

def display_note(text):
    display(HTML(f"<div class='sor-report sor-note'>{escape(str(text))}</div>"))

def display_table(title, frame, subtitle=""):
    display_section(title, subtitle)
    if frame is None or frame.empty:
        display_note("No rows were returned for this section.")
        return
    html = frame.to_html(index=False, escape=True).replace('class="dataframe"', 'class="sor-report-table"')
    display(HTML("<div class='sor-report'>" + html + "</div>"))

def display_kpis(cards):
    html = ''.join(f"<div class='sor-kpi'><span>{escape(str(label))}</span><strong>{escape(str(value))}</strong></div>" for label, value in cards)
    display(HTML("<div class='sor-report sor-kpi-grid'>" + html + "</div>"))

def badge(label, tone):
    css = {"pass":"sor-pass", "warn":"sor-warn", "fail":"sor-fail"}.get(tone, "sor-warn")
    return f"<span class='sor-badge {css}'>{escape(str(label))}</span>"

def first_value(frame, *names, default=0):
    for name in names:
        if name in frame.columns and len(frame) > 0:
            return frame.iloc[0][name]
    return default


BASE_URL = "http://127.0.0.1:8080"
TIMEOUT_SECONDS = 5.0
instrument_id = 0
side = "BUY"
quantity = 1000
urgency_id = 1

widget_controls = {}
if widgets is not None:
    widget_controls = {
        "base_url": widgets.Text(value=BASE_URL, description="API URL", layout=widgets.Layout(width="420px")),
        "timeout_seconds": widgets.FloatSlider(value=TIMEOUT_SECONDS, min=1.0, max=30.0, step=0.5, description="Timeout", continuous_update=False),
        "instrument_id": widgets.IntSlider(value=instrument_id, min=0, max=1, step=1, description="Instrument", continuous_update=False),
        "side": widgets.Dropdown(options=[("BUY", 1), ("SELL", 2)], value=1, description="Side"),
        "quantity": widgets.IntText(value=quantity, description="Quantity"),
        "urgency_id": widgets.Dropdown(options=[("LOW", 0), ("NORMAL", 1), ("HIGH", 2)], value=urgency_id, description="Urgency"),
    }
    display_styles()
    display(HTML("<div class='sor-report sor-section'><h2>Widget Control Panel</h2><p>Choose the parent order payload, then run the next cell to submit and report.</p></div>"))
    display(widgets.VBox([
        widgets.HBox([widget_controls["base_url"], widget_controls["timeout_seconds"]]),
        widgets.HBox([widget_controls["instrument_id"], widget_controls["side"], widget_controls["quantity"], widget_controls["urgency_id"]]),
    ]))
else:
    display_styles()
    display_note("ipywidgets is not installed. Install python/requirements.txt to enable friendly controls; defaults will be used.")

def read_widget_inputs():
    if widget_controls:
        return {
            "base_url": widget_controls["base_url"].value,
            "timeout_seconds": float(widget_controls["timeout_seconds"].value),
            "payload": {
                "instrumentId": int(widget_controls["instrument_id"].value),
                "side": int(widget_controls["side"].value),
                "quantity": int(widget_controls["quantity"].value),
                "urgencyId": int(widget_controls["urgency_id"].value),
            },
        }
    return {
        "base_url": BASE_URL,
        "timeout_seconds": TIMEOUT_SECONDS,
        "payload": {"instrumentId": instrument_id, "side": 1 if side == "BUY" else 2, "quantity": quantity, "urgencyId": urgency_id},
    }


## 2. Submit And Generate Report

Run the next cell to submit the order, fetch status, and render the report.

In [ ]:
inputs = read_widget_inputs()
BASE_URL = inputs["base_url"]
payload = inputs["payload"]
timeout_seconds = inputs["timeout_seconds"]
submitted_at = datetime.now(timezone.utc)

order = requests.post(f"{BASE_URL}/orders", json=payload, timeout=timeout_seconds).json()
status = requests.get(f"{BASE_URL}/orders/{order['parentOrderId']}", timeout=timeout_seconds).json()
completed_at = datetime.now(timezone.utc)

filled_qty = int(status.get("filledQty", 0))
remaining_qty = int(status.get("remainingQty", payload.get("quantity", 0)))
child_orders = status.get("childOrders", []) or []
outcome = "Completed" if remaining_qty == 0 else "Accepted with residual"
tone = "pass" if remaining_qty == 0 else "warn"

manifest = pd.DataFrame([{
    "baseUrl": BASE_URL,
    "submittedAtUtc": submitted_at.strftime("%Y-%m-%d %H:%M:%S"),
    "completedAtUtc": completed_at.strftime("%Y-%m-%d %H:%M:%S"),
    "timeoutSeconds": timeout_seconds,
}])
payload_frame = pd.DataFrame([payload])
order_frame = pd.DataFrame([order]).add_prefix("submit.")
status_frame = pd.DataFrame([{key: value for key, value in status.items() if key != "childOrders"}]).add_prefix("status.")
children_frame = pd.DataFrame(child_orders)
result_list = pd.DataFrame([
    {"Check": "Order submitted", "Result": "Yes", "Meaning": f"parentOrderId={order.get('parentOrderId')}"},
    {"Check": "Requested quantity", "Result": payload.get("quantity"), "Meaning": "Quantity sent to /orders"},
    {"Check": "Filled quantity", "Result": filled_qty, "Meaning": "Quantity routed/filled by the engine"},
    {"Check": "Remaining quantity", "Result": remaining_qty, "Meaning": "Residual quantity after routing"},
    {"Check": "Child orders", "Result": len(child_orders), "Meaning": "Venue-level child order count"},
])

what_happened = (
    f"Submitted a parent order for instrument {payload['instrumentId']} with quantity {payload['quantity']} "
    f"and urgency {payload['urgencyId']}. The engine returned parentOrderId {order.get('parentOrderId')}, "
    f"filled {filled_qty}, and left {remaining_qty} remaining."
)

display_styles()
display_header("Parent Order Submission Report", f"{BASE_URL}/orders", completed_at.strftime("%Y-%m-%d %H:%M:%S"))
display(HTML(f"<div class='sor-report'>{badge(outcome, tone)}</div>"))
display_kpis([
    ("Outcome", outcome),
    ("Parent order", order.get("parentOrderId", "")),
    ("Filled qty", filled_qty),
    ("Remaining qty", remaining_qty),
    ("Child orders", len(child_orders)),
])
display_note(what_happened)
display_table("1. Submission Manifest", manifest, "API endpoint and timing for this request.")
display_table("2. Parent Order Payload", payload_frame, "Exact payload sent with requests.post to /orders.")
display_table("3. Result List", result_list, "Plain-language checklist of the order result.")
display_table("4. Submit Response", order_frame, "Response returned by POST /orders.")
display_table("5. Order Status", status_frame, "Response returned by GET /orders/{parentOrderId}.")
display_table("6. Child Orders", children_frame, "Venue-level child order evidence, when available.")
